# 스마트 플래시카드 코치 V2 설계 및 테스트 셋업

## V2 주요 추가 사항
1. **Tool 연동**: 위키피디아(Wikipedia) 검색 도구를 연동하여 플래시카드 생성 시 외부 지식을 보강합니다.
2. **병렬 처리 (Send API)**: 추출된 여러 개의 개념을 LangGraph의 `Send`를 통해 병렬로 여러 노드로 뿌려 동시에 카드를 생성합니다.
3. **조건부 분기 (Conditional Edge)**: 채점 점수(Score)에 따라 80점 이상이면 종료, 미만이면 오답 힌트를 제공(`generate_hint`)하고 다시 퀴즈로 돌아가는 루프를 형성합니다.
4. **플래시카드 제한**: `flashcard_limit` 변수를 State에 도입하여 생성할 플래시카드의 최대 개수(기본값 5개)를 제어합니다.

## 그래프 흐름
```
[START] -> parse_material 
           ---(Send 병렬)---> generate_flashcard (Wiki Tool) 
           ---> aggregate_flashcards 
           ---> quiz_interrupt 
           ---> grade_quiz
           ---(Conditional Edge: Pass/Fail)---> [END] 또는 generate_hint -> quiz_interrupt
```

In [ ]:
# V2에 필요한 추가 패키지 설치
# %pip install wikipedia langchain-community

In [ ]:
import operator
from typing import TypedDict, List, Dict, Any
from typing_extensions import Annotated
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Send, Command
from langgraph.checkpoint.memory import InMemorySaver

from langchain.chat_models import init_chat_model
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# LLM 셋업
llm = init_chat_model("openai:gpt-4o-mini")

# 위키피디아 툴 셋업 (한국어 설정)
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="ko", top_k_results=1, doc_content_chars_max=1000))
llm_with_tools = llm.bind_tools([wiki_tool])

In [ ]:
# 1. State 및 Send용 구조체 정의

class State(TypedDict):
    learning_material: str
    concepts: List[str]
    # 병렬 병합을 위한 Reducer(operator.add) 적용
    flashcards: Annotated[List[Dict[str, str]], operator.add]
    user_answers: List[str]
    evaluation_report: str
    score: int
    hint: str
    flashcard_limit: int  # 플래시카드 최대 생성 제한 수

# Send API로 전달할 서브 상태 (병렬 노드용)
class FlashcardTask(TypedDict):
    concept: str

# 구조화된 출력을 위한 포맷 정의
class SingleFlashcard(BaseModel):
    question: str = Field(description="주어진 개념에 대해 묻는 한글 질문")
    answer: str = Field(description="위키피디아 등 검색된 배경지식을 포함한 명확한 개념 설명")

In [ ]:
# 2. V2 아키텍처용 노드 및 엣지 로직 구현

def parse_material(state: State) -> Dict[str, Any]:
    print("--- [노드] parse_material 실행 ---")
    material = state.get("learning_material", "")
    limit = state.get("flashcard_limit", 5)  # 설정된 생성 제한 수 (기본값 5)
    
    prompt = f"""
    학습 자료에서 핵심 교육 개념이나 전문 용어를 최대 {limit}개만 추출해 주세요.
    각 개념은 불필요한 설명이나 번호, 불릿 기호 없이 한 줄에 하나씩 명확한 단어나 핵심 어구로만 추출해 주세요.
    
    학습 자료:
    {material}
    """
    response = llm.invoke(prompt)
    concepts = [line.strip() for line in response.content.split("\n") if line.strip()]
    # 명확하게 지정된 개수만큼만 슬라이싱 제한
    concepts = concepts[:limit]
    return {"concepts": concepts}

def dispatch_flashcards(state: State):
    print(f"--- [라우팅] {len(state.get('concepts', []))}개의 개념을 Send로 병렬 분배 ---")
    return [Send("generate_flashcard", {"concept": c}) for c in state.get("concepts", [])]

def generate_flashcard(task: FlashcardTask) -> Dict[str, Any]:
    concept = task["concept"]
    print(f"--- [노드(병렬)] generate_flashcard: '{concept}' 처리 중 ---")
    
    # 위키피디아 툴을 활용한 배경지식 검색 연동
    try:
        wiki_info = wiki_tool.invoke({"query": concept})
    except Exception:
        wiki_info = "검색 결과 없음"
        
    prompt = f"""
    개념 '{concept}'에 대한 플래시카드를 작성하세요.
    위키피디아 검색 결과를 적극 참고하여 양질의 설명을 만드세요.
    
    위키피디아 검색 결과:
    {wiki_info}
    """
    
    structured_llm = llm.with_structured_output(SingleFlashcard)
    card_data = structured_llm.invoke(prompt)
    
    # 리스트 형태로 반환하여 operator.add에 의해 누적되게 함
    return {
        "flashcards": [{
            "question": card_data.question,
            "answer": card_data.answer
        }]
    }

def aggregate_flashcards(state: State) -> Dict[str, Any]:
    # 병렬 처리가 끝난 후 동기화(fan-in) 역할을 하는 더미 노드
    print("--- [노드] aggregate_flashcards: 모든 병렬 생성 취합 완료 ---")
    return {}

def quiz_interrupt(state: State) -> Dict[str, Any]:
    print("--- [노드] quiz_interrupt (대기) ---")
    # 이전 오답 힌트가 있다면 표시
    hint = state.get("hint", "")
    if hint:
        print(f"[시스템 힌트]: {hint}")
        
    flashcards = state.get("flashcards", [])
    answer = interrupt({
        "questions": [c["question"] for c in flashcards],
        "instruction": "답변 리스트를 'user_answers'에 담아 재개(resume)하세요."
    })
    return {"user_answers": answer.get("user_answers", [])}

def grade_quiz(state: State) -> Dict[str, Any]:
    print("--- [노드] grade_quiz ---")
    # 매우 단순화된 채점 로직 (실제로는 LLM 채점이 더 정확)
    # 데모를 위해 길이가 3글자 이상이면 정답 처리하여 점수 부여 (가정)
    user_answers = state.get("user_answers", [])
    correct_count = sum([1 for a in user_answers if len(str(a)) > 2])
    total = len(state.get("flashcards", []))
    score = int((correct_count / total) * 100) if total > 0 else 0
    
    report = f"### 채점 결과: {score}점\n"
    return {"evaluation_report": report, "score": score}

def check_score(state: State) -> str:
    score = state.get("score", 0)
    print(f"--- [조건] check_score: 현재 점수 {score}점 ---")
    if score >= 80:
        return "pass"
    return "fail"

def generate_hint(state: State) -> Dict[str, Any]:
    print("--- [노드] generate_hint (오답 루프 진입) ---")
    hint_msg = "점수가 80점 미만입니다. 개념의 뜻을 다시 한 번 자세히 생각해 보고 재도전하세요!"
    return {"hint": hint_msg}

In [ ]:
# 3. V2 그래프 빌드 및 조립
graph_builder = StateGraph(State)

graph_builder.add_node("parse_material", parse_material)
graph_builder.add_node("generate_flashcard", generate_flashcard)
graph_builder.add_node("aggregate_flashcards", aggregate_flashcards)
graph_builder.add_node("quiz_interrupt", quiz_interrupt)
graph_builder.add_node("grade_quiz", grade_quiz)
graph_builder.add_node("generate_hint", generate_hint)

graph_builder.add_edge(START, "parse_material")
# Send를 이용한 동적 팬-아웃 (병렬 라우팅)
graph_builder.add_conditional_edges("parse_material", dispatch_flashcards, ["generate_flashcard"])
# 팬-아웃된 병렬 처리들이 모두 끝나면 팬-인
graph_builder.add_edge("generate_flashcard", "aggregate_flashcards")
graph_builder.add_edge("aggregate_flashcards", "quiz_interrupt")
graph_builder.add_edge("quiz_interrupt", "grade_quiz")

# 조건부 엣지로 분기
graph_builder.add_conditional_edges("grade_quiz", check_score, {"pass": END, "fail": "generate_hint"})
# 실패 시 힌트를 주고 다시 퀴즈로 돌아가는 순환 구조
graph_builder.add_edge("generate_hint", "quiz_interrupt")

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

---
## 각 노드별 격리된 단위 테스트 구간
아래 셀들은 그래프 전체를 돌리지 않고 개별 함수(노드)를 단독 호출하여 동작을 검증하는 용도입니다.

In [ ]:
# [테스트 1] parse_material 테스트 (learning_material.txt 로드)
import os

file_path = "learning_material.txt"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        material_data = f.read()
else:
    material_data = "지도학습은 정답이 있는 데이터를 학습합니다.\n비지도학습은 정답이 없는 데이터를 학습합니다."

# 기본 flashcard_limit을 5개로 테스트 설정
test_state = {"learning_material": material_data, "flashcard_limit": 5}
res1 = parse_material(test_state)
print("추출된 개념:", res1)
assert "concepts" in res1, "concepts 키가 누락됨"

In [ ]:
# [테스트 2] generate_flashcard (Wiki Tool 연동 단독 테스트)
# [테스트 1](parse_material)에서 실제로 추출한 실데이터 개념 중 첫 번째 개념을 연동하여 테스트
if 'res1' in globals() and res1.get("concepts"):
    target_concept = res1["concepts"][0]
else:
    target_concept = "대한민국"

test_task = {"concept": target_concept}
print(f"--- 테스트 타겟 개념: '{target_concept}' ---")
res2 = generate_flashcard(test_task)
print("생성된 카드 (Tool 적용):", res2)
assert len(res2.get("flashcards", [])) == 1, "플래시카드 데이터 생성 실패"

In [ ]:
# [테스트 3] grade_quiz 및 check_score(Conditional Edge) 테스트
test_state3 = {
    "flashcards": [{}, {}], # 카드 2장 가정
    "user_answers": ["정답입니다", "아닙니다"] # 길이가 3 이상이므로 둘 다 정답 처리 됨 (데모 로직상)
}
res3 = grade_quiz(test_state3)
print("채점 결과:", res3)

condition_res = check_score({"score": res3["score"]})
print("분기 결정 결과:", condition_res)
assert condition_res == "pass", "100점인데 pass가 반환되지 않음"

In [ ]:
# 4. 통합 실행: 텍스트 파일 로드 및 첫 단계 실행
import os
config = {"configurable": {"thread_id": "notebook_session_korean"}}

file_path = "learning_material.txt"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        material_data = f.read()
else:
    material_data = "비디오 분석 기반 핵심 개념 요약본 (예시)"

# flashcard_limit을 기본값인 5로 주입하여 시작
initial_state = {"learning_material": material_data, "flashcard_limit": 5}
print("--- [통합 테스트 1] 그래프 최초 실행 시작 ---")
result = graph.invoke(initial_state, config=config)
print("\n인터럽트 중단 상태 도달. (사용자 퀴즈 답변 대기 중)")

In [ ]:
# 5. 사용자 답변 제출 및 그래프 재개 (resume) 테스트
# 인터럽트 대기 상태인 그래프에 임의의 학습자 답변 목록을 전달하여 남은 프로세스를 수행합니다.
from langgraph.types import Command

# 5개의 가상 답변 작성 (각 질문의 답으로 길이가 3 이상이므로 데모 채점 로직 통과 가정)
mock_user_answers = [
    "지도학습은 레이블이 있는 데이터로 모델을 학습시킵니다.",
    "비지도학습은 레이블이 없는 데이터에서 유용한 패턴을 찾습니다.",
    "강화학습은 보상을 최대화하는 에이전트의 결정을 학습합니다.",
    "분류 문제는 특정 그룹으로 데이터를 나누는 작업입니다.",
    "회귀 문제는 수치형 실숫값을 맞히는 예측 작업입니다."
]

print("--- [통합 테스트 2] 사용자 답변 전달 및 재개 ---")
config = {"configurable": {"thread_id": "notebook_session_korean"}}

final_result = graph.invoke(
    Command(resume={"user_answers": mock_user_answers}),
    config=config
)

print("\n=== 그래프 최종 완료 상태 ===")
print(f"퀴즈 최종 점수: {final_result.get('score')}점")
print(f"평가 보고서:\n{final_result.get('evaluation_report')}")
print(f"최종 힌트 상태: {final_result.get('hint', '없음 (통과)')}")